# MCMC Mathematical Foundation
This Juypter notebook will be covering the mathematical theory of MCMC in sequence of:
1. Markov Chain Basics: Four-Directional Proposal
2. Transition Matrix
3. Eigenvalues and Exponential Convergence
4. Metropolis Sampling
5. Visualisation 
6. Connection to the Simulation of $\pi$
7. Future Understanding

---

## Markov Chain Basics: Four-Directional Proposal
As mentioned in the mathematical basis for MCMC algorithms, a discrete four-directional random walk has a proposal probability, $q$ of $\frac{1}{4}$. 
This can be represented in Python with:

<div align="center">

```python 
neighbor[site][random.randint(0, 3)]
```
</div>

Furthermore, a `neighbour array` will be introduced to define the four possible destinations possible for each site in the 3 $\times$ 3 grid:

<div align="center">

```python
neighbour = [[1, 3, 0, 0], [2, 4, 0, 1], [2, 5, 1, 2],
            [4, 6, 3, 0], [5, 7, 3, 1], [5, 8, 4, 2],
            [7, 6, 6, 3], [8, 7, 6, 4], [8, 8, 7, 5]]
```
</div>

Each site in the 3x3 grid is represented by a number starting with 0 from the bottom-left corner, moving to the right and up, ending with 8 in the top-right corner:

<table align="center">
  <tr>
    <td>6</td>
    <td>7</td>
    <td>8</td>
  </tr>
  <tr>
    <td>3</td>
    <td>4</td>
    <td>5</td>
  </tr>
  <tr>
    <td>0</td>
    <td>1</td>
    <td>2</td>
  </tr>
</table>

The four numbers in each row represent the neighboring sites' numbers in the order: right, up, left, down.

From any site, each of the four possible proposal has an **equal** proposal probability:
```math 
q(i \rightarrow j) = \frac{1}{4}
```
The random walk can therefore be simulated by randomly selecting one of these four possibilities at each step.

Note: At the boundaries, a direction that would leave the grid points back to the current site. 
Thus, while each direction is proposed with probability 1/4, multiple directions may lead to the same destination, changing the transition probability.

In [ ]:
import random

# neighbour array for a 3x3 grid
neighbour = [[1, 3, 0, 0], [2, 4, 0, 1], [2, 5, 1, 2],
            [4, 6, 3, 0], [5, 7, 3, 1], [5, 8, 4, 2],
            [7, 6, 6, 3], [8, 7, 6, 4], [8, 8, 7, 5]]
t_max = 4
site = 8
t = 0
print(site)
while t < t_max:
    t += 1
    site = neighbour[site][random.randint(0, 3)]
    print(site)

8
5
2
5
5


## Transition Matrix 
The four-directional random walk can also be represented mathematically using a **transition matrix, $T$**, propagating the entire probability distribution.
The transition matrix constructed from the same `neighbour array` used previously, describes the complete set of one-step transition probabilities for the Markov chain.

For a Markov chain, the element $T_{ji}$ represents the probability of transitioning from state $i$ to state $j$ in one step. 
Since every directional proposal has an equal probability of $\frac{1}{4}$, and all proposed moves are accepted ($A=1$), using the relation $T=qA$:

```math
T_{ji} = \frac{1}{4} \times 1 = \frac{1}{4}

```

The transition matrix will show the propagation of the probability distribution towards the stationary distribution. 
With 9 geometrically equivalent sites in a 3 $\times$ 3 grid , the probability of occupying each site at equilibrium after a set amount of time, $t$ is

```math
P_\text{stationary}(i) = \frac{1}{9}

```

From that, it is also possible to calculate the deviation of each site's probability from uniformit / equilibrium with the following equation. 
As the Markov chain approaches equilibrium after a number of propagation steps $(t)$, $P_i(t) \text{approaches} \frac{1}{9}$, causing the deviation to approach zero as the propagation reaches a steady state:

```math
\lvert P_i(t) - \frac{1}{9} \rvert \rightarrow 0
```


In [7]:
import numpy as np

neighbor = [[1, 3, 0, 0], [2, 4, 0, 1], [2, 5, 1, 2],
            [4, 6, 3, 0], [5, 7, 3, 1], [5, 8, 4, 2],
            [7, 6, 6, 3], [8, 7, 6, 4], [8, 8, 7, 5]]

transfer = np.zeros((9, 9))
# creates a 9×9 matrix initially filled with zeroes because there are 9 possible states from site 8 to site 0.

for k in range(9):
    for neigh in range(4): 
        transfer[neighbor[k][neigh], k] += 0.25

def initialise():
    position = np.zeros(9)
    position[8] = 1.0
    return position

def propagate(transfer, n_steps):
    position = initialise()
    
    print('The propagation/evolution of the probability distribution at time, t')
    for t in range(n_steps):
        print(f't={t:<3}', ["%0.5f" % _ for _ in position])
        position = np.dot(transfer, position)

def deviation(transfer, n_steps):  
    position = initialise()

    print('\nThe deviation from uniformity/equilibrium at time, t')
    for t in range(n_steps):
        print(f't={t:<3}', ["%0.5f" % abs(i- 1.0 / 9.0) for i in position])
        position = np.dot(transfer, position)

def main():
    n_steps = 100
    propagate(transfer, n_steps)
    deviation(transfer, n_steps)

if __name__ == "__main__":
    main()

The propagation/evolution of the probability distribution at time, t
t=0   ['0.00000', '0.00000', '0.00000', '0.00000', '0.00000', '0.00000', '0.00000', '0.00000', '1.00000']
t=1   ['0.00000', '0.00000', '0.00000', '0.00000', '0.00000', '0.25000', '0.00000', '0.25000', '0.50000']
t=2   ['0.00000', '0.00000', '0.06250', '0.00000', '0.12500', '0.18750', '0.06250', '0.18750', '0.37500']
t=3   ['0.00000', '0.04688', '0.07812', '0.04688', '0.09375', '0.18750', '0.07812', '0.18750', '0.28125']
t=4   ['0.02344', '0.05469', '0.09766', '0.05469', '0.11719', '0.16016', '0.09766', '0.16016', '0.23438']
t=5   ['0.03906', '0.07324', '0.10254', '0.07324', '0.10742', '0.15234', '0.10254', '0.15234', '0.19727']
t=6   ['0.05615', '0.08057', '0.10767', '0.08057', '0.11279', '0.13989', '0.10767', '0.13989', '0.17480']
t=7   ['0.06836', '0.08929', '0.10895', '0.08929', '0.11023', '0.13379', '0.10895', '0.13379', '0.15735']
t=8   ['0.07883', '0.09421', '0.11024', '0.09421', '0.11154', '0.12758', '0.11024',

## Eigenvalues and Exponential Convergence
The transition matrix, $T$ determines how the distribution evolves with the propagation governed by its repeated application, giving the relation $p_{t} = T^t p_0$ after $t$ steps.
How do we understand that evolution as the distribution approaches the steady state?

The eigenvectors (2D Array / Matrix) characterize corresponding modes of the Markov chain's evolution, while the eigenvalues (1D Array) determine how these modes evolve with repeated application of the transition matrix. 
The stationary distribution is an eigenvector of the transition matrix T with an eigenvalue of 1, meaning that it remains unchanged when the transition matrix is applied.
For a given matrix $A$, the fundamental eigenvalue equation determines $v$ and the corresponding $λ$:
```math
Av = \lambda v
```
where:
- $v$ is an eigenvector whose direction remains unchanged when transformed by a square matrix.
- $\lambda$ is the eigenvalue that determines if $v$ stretches, shrinks, or flips.

Hence, for the eigenvector corresponding to the dominant eigenvalue of 1, the equation evolves to $T^t v = v$:
```math
T^t v = \lambda^t v = 1^t \cdot v = v
```
The remaining eigenvalues with magnitude $\lvert \lambda \rvert < 1$ govern how fast the system reaches equilibrium, giving a measurement of the `exponential` convergence rate. 
This means the non-equilibrium states will decay to zero exponentially as time, $t \rightarrow \infty$, approaching the stationary probability ($P_\text{stationary}(i) = \frac{1}{9}$) with
```math 
\lvert \lambda^t \rvert \rightarrow 0
```

This decay also highlights that the largest magnitude below 1, the subdominant eigenvalue, determines the slowest-decaying component and therefore controls the asymptotic convergence rate.

### Eigenvalues Python Script 

The eigenvalues and eigenvectors that provide the mathematical basis for the observed convergence towards equilibrium can be determined in Python with:
```python
import numpy as np

# Continuing from the previous code
def eigen():
    eigenvalues, eigenvectors = np.linalg.eig(transfer)
    print(np.real_if_close(eigenvalues))
```

E.g., continuing from the previous `neighbour array`, the Python function above will output:

<table align="center">
  <tr>
    <td>-5.00000000e-01</td>
    <td> 1.00000000e+00</td>
    <td> 5.00000000e-01</td>
  </tr>
  <tr>
    <td> 7.50000000e-01</td>
    <td> 7.50000000e-01</td>
    <td>-1.10578001e-16</td>
  </tr>
  <tr>
    <td>-1.75540133e-17</td>
    <td> 2.50000000e-01</td>
    <td> 2.50000000e-01</td>
  </tr>
</table>

In this case, $\lambda = 1$ shows the stationary component remains unchanged, while $\lvert \lambda \rvert = 0.75$ controls the asymptotic convergence rate.

---

## Metropolis Sampling

The previous sections considered a Markov chain with equal transition probabilities between neighbouring sites. 
However, for actual implementation of MCMC sampling, the transition / acceptance mechanism has to be designed so that some states are visited more frequently.

Continuing from the previous `neighbour array`, the modification can be done with a set of weights proportional to the probability of a state $P(i) \propto w_i$:

```python
weight = [3.0, 0.5, 1.0, 0.5, 1.0, 0.5, 2.0, 0.5, 1.0]
```

The randomly proposed move is then accepted according to the ratio of the weight of the proposed state to the current state, matching the Metropolis acceptance probability:

```math
A(i \rightarrow j) = \min\left(1,\frac{w_j}{w_i}\right) = \min\left(1,\frac{P(j)}{P(i)}\right)
```

If the proposed state has a greater weight than the current state, the move is always accepted ($\text{ratio} \geq 1$). 
Meanwhile, the move is accepted with the corresponding probability of that particular $A(i \rightarrow j)$ if the proposed state has a lower weight.
This allows the chain to occasionally move towards lower-probability states while favouring higher-probability states. 
After many iterations, the histogram of visited states approaches the target distribution defined by the weights.
The histogram can therefore be compared with the relative weights to verify that the Markov chain is sampling from the intended distribution.

In [13]:
import random

neighbor = [[1, 3, 0, 0], [2, 4, 0, 1], [2, 5, 1, 2],
            [4, 6, 3, 0], [5, 7, 3, 1], [5, 8, 4, 2],
            [7, 6, 6, 3], [8, 7, 6, 4], [8, 8, 7, 5]]

weight = [3.0, 0.5, 1.0, 0.5, 1.0, 0.5, 2.0, 0.5, 1.0]

def metropolis(n_iter):
    histo = [0] * 9
    pos = 8

    for _ in range(n_iter):
        new_pos = neighbor[pos][random.randint(0, 3)]

        if random.random() < weight[new_pos] / weight[pos]:
            pos = new_pos
        # Metropolis acceptance probability
        
        histo[pos] += 1

    return histo

def main():
    n_iter = 1_000_000
    histo = metropolis(n_iter)

    norm = sum(weight)

    for k in range(9):
        print(f'site: {k}', f'weight: {weight[k]}', f'histo: {norm * histo[k] / float(n_iter)}')

if __name__ == "__main__":
    main()

site: 0 weight: 3.0 histo: 2.99648
site: 1 weight: 0.5 histo: 0.50067
site: 2 weight: 1.0 histo: 1.01162
site: 3 weight: 0.5 histo: 0.50248
site: 4 weight: 1.0 histo: 1.00606
site: 5 weight: 0.5 histo: 0.49809
site: 6 weight: 2.0 histo: 1.98366
site: 7 weight: 0.5 histo: 0.50271
site: 8 weight: 1.0 histo: 0.99823


## Visualisation of MCMC Algorithms
What does a Markov chain actually look like as it evolves? Random movement of an individual chain(single trajectory)? Propagating probability distribution (repeated sampling)? 

1. Single-chain visualisation

    For the 3 $\times$ 3 grid system, the animation follows the position of a single pebble as it moves between neighbouring sites. 
    Each step depends only on its current position and the randomly selected neighbouring direction, providing an intuitive representation of a Markov chain trajectory as the chain "wanders" around the state space.

In [ ]:
import random, matplotlib.pyplot as plt

sigma = 0.4  # sigma and s_map are needed for the graphical output
s_map = [(1.0, 1.0), (2.0, 1.0), (3.0, 1.0), 
         (1.0, 2.0), (2.0, 2.0), (3.0, 2.0), 
         (1.0, 3.0), (2.0, 3.0), (3.0, 3.0)] 
neighbor =  [[1, 3, 0, 0], [2, 4, 0, 1], [2, 5, 1, 2],
             [4, 6, 3, 0], [5, 7, 3, 1], [5, 8, 4, 2],
             [7, 6, 6, 3], [8, 7, 6, 4], [8, 8, 7, 5]]
site = 8
N_runs = 10
for run in range(N_runs):
    if run < 10: number_string = '0'+str(run)
    else: number_string = str(run)
    # Begin of graphical output
    cir = plt.Circle(s_map[site], radius=sigma, fc='r')
    plt.gca().add_patch(cir)
    plt.plot([0.5, 3.5], [1.5, 1.5], 'b')
    plt.plot([0.5, 3.5], [2.5, 2.5], 'b')
    plt.plot([1.5, 1.5], [0.5, 3.5], 'b')
    plt.plot([2.5, 2.5], [0.5, 3.5], 'b')
    plt.title('t = '+ number_string)
    plt.axis('scaled')
    plt.axis([0.5, 3.5, 0.5, 3.5])
    plt.xticks([])
    plt.yticks([])
    plt.savefig('pebble_basic_movie_'+number_string+'.png', transparent=False)
    plt.show()
    plt.clf()
    # End of graphical output
    site = neighbor[site][ random.randint(0, 3)]

2. Two-state visualisation

    The two-state example provides a simpler representation of Markov-chain transitions. 
    With only two possible states, the effect of the transition probabilities can be observed directly, providing a visual connection to the two-state eigenvalue example from the previous section.

In [ ]:
import math, random, matplotlib.pyplot as plt

sigma = 0.4
epsilon = 0.1
plt.figure()
s_map = [(1.0, 1.0), (2.0, 1.0)] 
neighbor =  [[1], [0]]
pos = 0
tmax = 20
for iter in range(tmax):
    # Begin of the graphics output
    plt.figure()
    number_string = str(iter).zfill(len(str(tmax)))
    cir = plt.Circle(s_map[pos], radius=sigma, fc='r')
    plt.gca().add_patch(cir)
    plt.plot([1.5, 1.5], [0.5, 1.5], 'b')
    plt.title('t = '+ number_string)
    plt.axis('scaled')
    plt.axis([0.5, 2.5, 0.5, 1.5])
    plt.xticks([])
    plt.yticks([])
    plt.savefig('2x1pebble_epsilon'+number_string+'.png', transparent=True)
    plt.close()
    # End of the graphics output
    newpos = neighbor[pos][0]
    if random.random() < epsilon:
        newpos = pos
    pos = newpos

3. Multiple-chain histogram

    When many independent chains are sampled, the individual trajectories become less important than the overall distribution of visited states. 
    A two-dimensional histogram can be used to show how frequently each site is occupied, demonstrating how repeated Markov-chain sampling can recover the underlying probability distribution.

    Moreover, the histogram can be repeated at different propagation times to show the distribution evolution from its initial state toward the stationary distribution. 
    Implementing in the previous 3 $\times$ 3 grid system, the histogram will gradually become approximately uniform across the nine sites as the number of propagation steps increases, approaching the equilibrium distribution of $P_\text{stationary}(i)=\frac{1}{9}$.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

xvec = {1:3, 2:2, 3:1, 4:3, 5:2, 6:1, 7:3, 8:2, 9:1} 
yvec = {1:1, 2:1, 3:1, 4:2, 5:2, 6:2, 7:3, 8:3, 9:3} 

neighbor = {1 : [2, 4, 1, 1], 2 : [3, 5, 1, 2], 3 : [3, 6, 2, 3],
            4 : [5, 7, 4, 1], 5 : [6, 8, 4, 2], 6 : [6, 9, 5, 3],
            7 : [8, 7, 7, 4], 8 : [9, 8, 7, 5], 9 : [9, 9, 8, 6]}

def histogram(run, n_samples):
    list_vec = []

    for _ in range(n_samples): 
        pos = 9

        for _ in range(run):
            pos = neighbor[pos][random.randint(0, 3)]

        list_vec.append(pos)

    x = [xvec[k] for k in list_vec]
    y = [yvec[k] for k in list_vec]

    plt.xticks([])
    plt.yticks([])

    H, xedges, yedges = np.histogram2d(
        x, y, bins=(3, 3), range=[[1, 3], [1, 3]]
    )

    H /= np.sum(H)

    print(H)

    extent = [yedges[0], yedges[-1], xedges[-1], xedges[0]]

    histo = plt.imshow(H, extent=extent, interpolation='nearest', vmin=0, vmax=1)
    histo.set_cmap('hot')
    plt.colorbar()
    plt.title('t = ' + str(run), fontsize=22)
    plt.savefig(f'3x3_pebble_run_{run:02d}.png')
    plt.show()
    plt.close()
    

def main():
    n_samples = 100_000

    for run in range(10):
        histogram(run, n_samples)

if __name__ == "__main__":
    main()

## Connection to the Simulation of $\pi$

In conclusion, `Markov chain Monte Carlo` calculations can be seen to be equivalent to `transfer matrix` multiplication. 
Furthermore, although a steady state / equilibrium will be reached after an infinite number of iterations, it is not a major restriction due to the existence of a time scale (set by the subdominant eigenvalue).
The resulting statistical error can be further quantified using the powerful `bunching algorithm`, which accounts for correlations between successive samples in the Markov chain

The Markov-chain concepts demonstrated in the previous sections form the basis of the MCMC approach used in the $\pi$ simulation. 
- The simplified representation of how a Markov chain evolves toward a target distribution (discrete 3 $\times$ 3 grid) are also the same principles applied in the as the continuous two-dimensional space ($\pi$ simulation).
- The Metropolis sampling example is particularly relevant, as the same acceptance mechanism is used to generate the continuous chain used in the estimation of $\pi$ from the proportion of sampled states lying within the unit circle.

---

## Future Understanding: Dual-Pebbles of our Adult Monte Carlo Game and Other Concepts
Last but not least, a Markov chain can contain multiple regions of state space that are connected by transitions. 
For the chain to explore the full state space, transitions between these regions must be possible.

### Disconnected Regions and Reducibility

If the state space is separated into two or more regions with no possible transitions between them, the Markov chain is reducible.

```math
T = \begin{pmatrix} T_1 & 0\\ 0 & T_2 \end{pmatrix}. 
```

The transition matrix can be separated into independent subsystems, each with its own stationary distribution and corresponding eigenvector with eigenvalue 1. 
Thus, the number of independent stationary distributions is reflected by the multiplicity of the eigenvalue 1. 
For a Markov chain intended to explore the entire state space, this behaviour is undesirable because the sampled distribution can depend on the region in which the chain was initially placed. 
Consequently, the outcome of a reducible MCMC simulation will depend on the starting configutation, even at infinte time, $t$.

### Periodicity and Oscillation

Periodic behaviour in a Markov chain is caused by eigenvalues with magnitude = 1 ($\lvert \lambda \rvert = 1$) other than the stationary eigenvalue 1 ($\lambda_1 = 1$). 
An eigenvalue of `-1` produces a component which changes sign at every step, causing an oscillation of the probability vector that never converges to its stationary distribution, even at infinite time, $t$.

```math
(-1)^t
```

This creates a periodic chain, where the constrained states follow a repeating cycle indefinitely instead of exploring the entire state space (*undesirables*).

```math
A\rightarrow B\rightarrow A\rightarrow B\rightarrow\cdots
```

A simple way to avoid strict periodicity is to allow a chain to remain in its current state with non-zero probability. 
- In the 3 × 3 grid, this can naturally occur at the boundaries, where the probability for directions leading outside the grid allows the pebble to remain at its current position. 
- This is also evident in the implementation of MCMC in the *$\pi$ estimation* as proposal that lies outside the unit circle are continuously rejected and `stacked` to its previous position.
These self-transitions help prevent the chain from becoming trapped in a fixed cycle, enforcing aperiodicity.

### Conditions for a Well-Behaved Markov Chain

1. Condition 0: Global Balance

    The stationary distribution must satisfy global balance as introduced intially ($T^t v = v$), possessing an eigenvalue of 1 to ensure the probability distribution remains unchanged after a transition. 

    ```math
    Tp_{\mathrm{stationary}}=p_{\mathrm{stationary}}.
    ```

2. Condition 1: Irreducibility

    As mentioned previously, a irreducible Markov chain is required to allow every relevant state can eventually be reached from every other relevant state. 
    This prevents the state space from being separated into disconnected regions and allows the chain to explore the complete state space.

3. Condition 2: Aperiodicity

    Aperiodicity prevents persistent oscillations and allows the distribution to converge to its stationary distribution at a fixed period rather than constrained to repeatedly cycle between different distributions. 

Together, these conditions help ensure that a Markov chain can properly explore its state space and converge to a unique stationary distribution under the appropriate assumptions.



### Dual-Pebble Example

By extending the 3 × 3 grid into two coupled region with a system of 18 possible states, an appriorate 18 × 18 transfer matrix example to examine the conditions can be created.

The `dual_eigen` example calculates the eigenvalues of this larger transition matrix.
If the two regions were completely disconnected, they would form separate subsystems and the chain would be reducible, with multiple eigenvalues equal to 1.

By introducing transitions between the two regions, the complete system becomes connected and the markov chain will explore both regions rather than stuck to one region. 
This demonstrates how a chain can be irreducible while still mixing slowly.

The eigenvalues provide a way to describe this behaviour. As discussed previously, non-stationary components decay according to powers of their corresponding eigenvalues. Therefore, an eigenvalue whose magnitude is close to 1 corresponds to a slowly decaying component and consequently slow convergence or mixing.

The corresponding `dual_movie` visualises this behaviour, showing how the pebble can remain within one region for an extended period before transitioning to the other.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
 
neighbor =  [[1, 3, 0, 0], [2, 4, 0, 1], [2, 5, 1, 2],
             [4, 6, 3, 0], [5, 7, 3, 1], [5, 8, 4, 2],
             [7, 6, 6, 3], [8, 7, 6, 4], [8, 8, 7, 5]]

def dual_eigen():
    transfer = np.zeros((18, 18))
    for k in range(9):
        for neigh in range(4): 
            transfer[neighbor[k][neigh], k] += 0.25     # red pebble game
            transfer[neighbor[k][neigh]+9, k+9] += 0.25 # blue pebble game   
    # small transition epsilon between red 2 and blue 6

    epsilon = 0.04
    transfer[6+9,2] = transfer[2,6+9] = epsilon
    transfer[2,2] -= epsilon
    transfer[6+9,6+9] -= epsilon
    eigenvalues, eigenvectors = np.linalg.eig(transfer)
    print (np.real_if_close(eigenvalues))

def dual_movie():
    rng = np.random.default_rng(seed=1234)

    sigma = 0.4
    epsilon = 0.4  # probability to switch from red to blue pebble, and vice versa (larger than in dual_eigen() for clearer visualisation).
    period = 4
    offset = 3.
    tmax = 240

    s_map_red =  [(1.0, 1.0), (2.0, 1.0), (3.0, 1.0), 
                (1.0, 2.0), (2.0, 2.0), (3.0, 2.0), 
                (1.0, 3.0), (2.0, 3.0), (3.0, 3.0)] 

    s_map_blue = [(x+offset,y-offset) for (x,y) in s_map_red]

    color = 'red'  #chose 'red' or 'blue'
    site = 8
    
    for iter in range(tmax):
        if (iter%period) == 0:
            # Begin of graphical output
            maxlength = len(str(tmax-1))
            number_string = str(iter).zfill(maxlength)
            if color == 'red':  cir = plt.Circle(s_map_red[site],  radius=sigma, fc='r')
            else:cir = plt.Circle(s_map_blue[site], radius=sigma, fc='b')
            plt.figure()
            plt.gca().add_patch(cir)

            # Draw red and blue grids
            plt.plot([0.5, 3.5], [0.5, 0.5], 'r')
            plt.plot([0.5, 3.5], [1.5, 1.5], 'r')
            plt.plot([0.5, 3.5], [2.5, 2.5], 'r')
            plt.plot([1.5, 1.5], [0.5, 3.5], 'r')
            plt.plot([2.5, 2.5], [0.5, 3.5], 'r')
            plt.plot([3.5, 3.5], [0.5, 3.5], 'r')
            plt.plot([0.5+offset, 3.5+offset], [1.5-offset, 1.5-offset], 'b')
            plt.plot([0.5+offset, 3.5+offset], [2.5-offset, 2.5-offset], 'b')
            plt.plot([0.5+offset, 3.5+offset], [3.5-offset, 3.5-offset], 'b')
            plt.plot([0.5+offset, 0.5+offset], [0.5-offset, 3.5-offset], 'b')
            plt.plot([1.5+offset, 1.5+offset], [0.5-offset, 3.5-offset], 'b')
            plt.plot([2.5+offset, 2.5+offset], [0.5-offset, 3.5-offset], 'b')

            plt.title('t = '+ number_string)
            plt.axis('scaled')
            plt.axis([0.5, 6.5, -2.5, 3.5])
            plt.xticks([])
            plt.yticks([])

            number_string_filename = str(iter // period).zfill(3)
            plt.savefig('pebble_dual_movie_epsilon_'+number_string_filename+'.png', transparent=True)
            plt.close()
            # End of graphical output
        newsite = neighbor[site][ rng.integers(0, 4)]
        newcolor = color

        if (color == 'red') and (site == 2) and (newsite == 2):
            if rng.random() < epsilon:
                newcolor = 'blue'
                newsite = 6
                print ("transition red->blue at time = ", iter)
        if (color == 'blue') and (site == 6) and (newsite == 6):
            if rng.random() < epsilon:
                newcolor = 'red'
                newsite = 2
                print ("transition blue->red at time = ", iter)

        site = newsite
        color = newcolor

def main():
    dual_eigen()
    dual_movie()

if __name__ == "__main__":
    main()

### Further MCMC Concepts

The examples above introduce the basic mechanisms behind Markov-chain sampling, but several additional concepts are important when evaluating the quality and efficiency of an MCMC simulation:
1. Ergodicity — the conditions under which long-run averages obtained from a Markov chain can represent averages over the target distribution.
2. Burn-in — the initial portion of a chain that may be discarded before collecting samples for estimation.
3. Autocorrelation — the statistical dependence between successive samples in a chain.
4. Mixing time — a measure of how long a chain takes to sufficiently approach its stationary distribution.
5. Proposal step-size and acceptance-rate tradeoffs — how the size of proposed moves affects both the acceptance rate and efficiency of exploration.
6. Asymmetric Metropolis-Hastings proposals — the more general case where the probability of proposing a transition in one direction differs from the reverse transition.

These concepts provide the next level of understanding for assessing the convergence, efficiency, and reliability of MCMC simulations beyond the examples demonstrated in this notebook.